# Portfolio Optimization with QAOA

> 🚀 **Don't choke your local machine.** Qoro is giving away **$100 in free cloud compute credits.**
> Get your API key at **[dash.qoroquantum.net](https://dash.qoroquantum.net)** to run this at scale.

## Why Cloud?

A portfolio of **480 assets** gets partitioned into dozens of sub-problems, each requiring its own QAOA optimization. Running these sequentially takes **hours**. QoroService runs every partition **in parallel** — all portfolios optimized simultaneously.

## Step 0 — Install & Authenticate

```bash
pip install qoro-divi numpy pandas
```

In [ ]:
import os
import time

import numpy as np

# Set your API key (get one free at https://dash.qoroquantum.net)
# Create a .env file at the root and set the QORO_API_TOKEN

from divi.backends import QoroService, ParallelSimulator, JobConfig

## Overview

Portfolio optimization selects assets to **maximize return while minimizing risk**. This is formulated as a QUBO:

$$\text{Minimize: } x^T \Sigma x - \lambda \mu^T x$$

where $\Sigma$ is the covariance matrix, $\mu$ is the returns vector, and $\lambda$ balances risk vs. return.

For large portfolios, the problem is partitioned using **spectral graph partitioning** based on asset correlations.

---

## Phase 1 — Small Portfolio (Local)

Generate synthetic data for a small portfolio to prove the workflow.

In [ ]:
# Generate synthetic financial data
np.random.seed(42)
n_assets = 8

# Random returns and covariance
returns = np.random.uniform(0.01, 0.1, n_assets)
A = np.random.randn(n_assets, n_assets) * 0.1
covariance = A @ A.T + np.eye(n_assets) * 0.01  # Positive definite

print(f"📊 Portfolio: {n_assets} assets")
print(f"   Returns range: [{returns.min():.4f}, {returns.max():.4f}]")
print(f"   Covariance shape: {covariance.shape}")

In [ ]:
from utils import build_qubo_matrices_for_all_partitions, evaluate_solution
import dimod

from divi.qprog import QAOA
from divi.qprog.optimizers import MonteCarloOptimizer

LAMBDA_PARAM = 0.75

# Build QUBO for single partition (whole portfolio)
qubo_matrices = build_qubo_matrices_for_all_partitions(
    {0: returns}, {0: covariance}, lambda_param=LAMBDA_PARAM
)
qubo_matrix = qubo_matrices[0]

# Convert to BQM
bqm = dimod.BinaryQuadraticModel(qubo_matrix, "BINARY")
print(f"QUBO: {len(bqm.variables)} variables")

In [ ]:
# Solve with QAOA locally
local_backend = ParallelSimulator(shots=10_000)
optim = MonteCarloOptimizer(population_size=30, n_best_sets=5)

print("🚀 Running QAOA locally...")
t0 = time.time()

qaoa = QAOA(
    bqm,
    n_layers=2,
    optimizer=optim,
    max_iterations=10,
    backend=local_backend,
)
qaoa.run()

local_time = time.time() - t0
print(f"✅ QAOA complete in {local_time:.1f}s")

In [ ]:
# Extract and evaluate solution
qaoa_solution = np.array(qaoa.solution, dtype=int)
print(f"QAOA solution: {qaoa_solution}")
print(f"Assets selected: {np.where(qaoa_solution == 1)[0]}")

evaluate_solution(qubo_matrix, qaoa_solution, returns, covariance)

---

## Phase 2 — Scale Up with QoroService

For production-scale portfolios (100+ assets), use spectral partitioning + QoroService:

```python
from divi.backends import QoroService, JobConfig
cloud_backend = QoroService(job_config=JobConfig(shots=10_000))

# Use GraphPartitioningQAOA for automatic partitioning
from divi.qprog import QUBOPartitioningQAOA
```

👉 **[Get your free API key →](https://dash.qoroquantum.net)**

---

## Ready for Real-World Portfolios?

480 assets, dozens of partitions, all optimized in parallel.

👉 **[Get your free API key at dash.qoroquantum.net](https://dash.qoroquantum.net)** — $100 in credits, no credit card required.